# 3. Incidents, Investigation, and Automated Response

### What you'll learn
- How alerts correlate into incidents (the SOC unit of work)
- The four-step investigation workflow (triage -> investigate -> respond -> close)
- The bad -> best progression from manual response to automated playbooks
- How threat-intelligence watchlists add context to investigations

## From alerts to incidents

A single real attack produces **many alerts** across products. The SIEM correlates them into ONE incident:

```
Alert: Brute force sign-in (alice)            -+
Alert: Sign-in from suspicious location        +--> Incident: Compromised account (alice)
Alert: Lateral movement from alice's laptop   -+
```

## 0. Setup — pick the lab kernel

This lab has its own `uv`-managed virtual environment. Before running any code cell:

1. From `security-certs/sc-200/01-build-a-siem/` run once in a terminal:
   ```bash
   uv sync
   docker compose up -d
   ```
2. In VS Code, click the kernel picker (top-right of this notebook) and choose **`.venv (Python 3.xx)`** from this folder.
3. If the kernel does not appear, reload the window: `Cmd+Shift+P` → `Reload Window`.

The log-generator container has already seeded the SIEM with normal traffic **and** four attack patterns (brute force, lateral movement, exfiltration, phishing). Every cell below talks to `http://localhost:8000`.

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

incidents = httpx.get(f'{SIEM}/incidents').json()
print(f'=== Open Incidents ({len(incidents)}) ===\n')
for inc in incidents:
    sev = {'Critical':'🟣','High':'🔴','Medium':'🟡','Low':'🟢'}.get(inc['severity'], '⬜')
    alerts = json.loads(inc['alert_ids']) if isinstance(inc['alert_ids'], str) else inc['alert_ids']
    entities = json.loads(inc['entities']) if isinstance(inc['entities'], str) else inc['entities']
    print(f'{sev} {inc["id"]} [{inc["status"]}]  {inc["title"]}')
    print(f'   Alerts: {len(alerts)}  |  Entities: {entities}  |  Assigned: {inc["assigned_to"] or "Unassigned"}\n')

## 3.1 Investigation workflow

1. **Triage** — severity OK? real? assign an analyst.
2. **Investigate** — drill into alerts, entities, timeline.
3. **Contain & Remediate** — disable users, isolate devices, block IPs.
4. **Close** — classify (TruePositive / BenignPositive / FalsePositive / Undetermined) and document.

In [ ]:
# Find a High-severity incident that has a real entity to pivot on
high_incidents = [i for i in incidents if i['severity'] == 'High']
target = next(
    (i for i in high_incidents if (json.loads(i['entities']) if isinstance(i['entities'],str) else i['entities'])),
    high_incidents[0] if high_incidents else None,
)

if target:
    details = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
    print(f'=== Investigating: {details["id"]} ===')
    print(f'Title: {details["title"]}')
    print(f'Severity: {details["severity"]}  |  Status: {details["status"]}')
    print(f'\n--- Related alerts ({len(details["alerts"])}) ---')
    for alert in details['alerts']:
        ev = json.loads(alert['evidence']) if isinstance(alert['evidence'], str) else alert['evidence']
        print(f'  🚨 {alert["title"]}')
        print(f'     Tactic: {alert["tactic"]}  |  Severity: {alert["severity"]}')
        if ev: print(f'     Evidence[0]: {json.dumps(ev[0], indent=2)[:200]}')
        print()

In [ ]:
# Step 1 - Triage: assign, activate, comment
if target:
    httpx.patch(f'{SIEM}/incidents/{target["id"]}', json={
        'status': 'Active',
        'assigned_to': 'soc-analyst-1@contoso.com',
        'comment': 'Triaged - looks like a real brute force, investigating.',
    })
    updated = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
    print(f'Status: {updated["status"]}  |  Assigned: {updated["assigned_to"]}')
    comments = json.loads(updated['comments']) if isinstance(updated['comments'], str) else updated['comments']
    print('Comments:', comments)

In [ ]:
# Step 2 - Investigate: pivot from entity to all related activity
if target:
    entities = json.loads(target['entities']) if isinstance(target['entities'], str) else target['entities']
    if 'UserPrincipalName' in entities:
        user = entities['UserPrincipalName']
        print(f'--- Sign-in breakdown for {user} ---')
        r = httpx.post(f'{SIEM}/query', json={'table_name':'SigninLogs','filter':{'UserPrincipalName':user},'aggregate_by':'ResultType'})
        for row in r.json()['results']:
            print(f'  {row["group_key"]}: {row["count"]}')
        print(f'\n--- Endpoint activity for {user.split("@")[0]} ---')
        r = httpx.post(f'{SIEM}/query', json={'table_name':'DeviceEvents','filter':{'AccountName':user.split("@")[0]},'limit':10})
        for log in r.json()['results']:
            flag = '⚠️' if log.get('FileName') in ('mimikatz.exe','psexec.exe') else '  '
            print(f'  {flag} {log["DeviceName"]}: {log["FileName"]} ({log["ActionType"]})')

## 3.2 Bad response -> Best response (automation)

Manual response is slow. At 3am, it also does not happen. Mature SOCs encode the runbook into a **playbook** that runs automatically.

| Approach | Mean-time-to-respond | Reliability |
|---------|---------------------|-------------|
| 🔴 Manual (analyst logs in, clicks through portals) | Minutes to hours | Low — depends on staffing |
| 🟡 Scripted one-off (Python script per alert type) | Minutes | Medium — scripts drift |
| ✅ **Playbook** (Logic App triggered by the SIEM) | Seconds | High — audited, version-controlled |

In [ ]:
# BAD: print what an analyst WOULD do manually for this incident.
print('Manual runbook (what an analyst would do by hand):')
print('  1. Open Entra admin portal')
print('  2. Find the user')
print('  3. Disable the account')
print('  4. Revoke sessions')
print('  5. Email the SOC channel')
print('Typical time: 5-15 minutes, IF the analyst is awake.')

In [ ]:
# BEST: trigger a playbook. One API call, executes every step instantly.
if target:
    r = httpx.post(f'{SIEM}/playbooks/run/{target["id"]}').json()
    if r['playbooks_executed']:
        for pb in r['playbooks_executed']:
            print(f'🤖 Playbook: {pb["playbook"]}')
            for i, action in enumerate(pb['actions'], 1):
                print(f'   Step {i}: ✅ {action["description"]}')
    else:
        print('No matching playbooks. Check severity/tactic match in /playbooks.')
print('\n💡 In real Sentinel, playbooks are Logic Apps that call Azure APIs:')
print('   - Disable user   -> Microsoft Graph API')
print('   - Isolate device -> Defender for Endpoint API')
print('   - Block IP       -> Azure Firewall API')
print('   - Notify SOC     -> Teams / ServiceNow webhook')

## 3.3 Threat-intelligence watchlists (IOC matching)

A **watchlist** is a lookup table of known-bad (or known-good) values you import into the SIEM. Typical uses:

- 🕷️ Known-bad IPs from a threat feed (e.g., AlienVault OTX, Microsoft TI)
- 👑 VIP users who need extra monitoring
- 🧑‍💻 Terminated employees (any activity = urgent)
- 🗝️ Approved admin tools (allowlist)

In KQL the pattern is `Table | where Column in (watchlist)`. Let's do it in our mini-SIEM.

In [ ]:
# Create a TI watchlist of known-bad IPs
httpx.post(f'{SIEM}/watchlists', json={
    'name': 'known_bad_ips',
    'description': 'Simulated threat-intel feed (Tor exits, C2 servers)',
    'items': ['185.220.101.42', '45.33.32.156', '198.51.100.99'],
})

# Match the watchlist against sign-in logs
r = httpx.post(f'{SIEM}/watchlists/match', json={
    'watchlist': 'known_bad_ips',
    'table_name': 'SigninLogs',
    'field': 'IPAddress',
    'time_range_minutes': 1440,
}).json()
print(f'🔍 {r["match_count"]} sign-ins from known-bad IPs:')
for m in r['matches'][:5]:
    print(f'  {m["UserPrincipalName"]:<20} {m["IPAddress"]:<16} {m["ResultType"]:<8} {m["Location"]}')
print('\n💡 Real Sentinel ships a free Microsoft TI feed plus the ThreatIntelligenceIndicator table.')

In [ ]:
# Step 4 - Close the incident with a classification
if target:
    httpx.patch(f'{SIEM}/incidents/{target["id"]}', json={
        'status': 'Closed',
        'classification': 'TruePositive',
        'comment': 'Confirmed brute force. Account disabled via playbook. Password reset required.',
    })
    final = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
    print(f'Status: {final["status"]}  |  Classification: {final["classification"]}')
    for c in (json.loads(final['comments']) if isinstance(final['comments'], str) else final['comments']):
        print(f'  - {c["text"]}')

print('\n--- Incident classifications ---')
print('  TruePositive   - confirmed attack, action taken')
print('  BenignPositive - real activity, not malicious (e.g., pen test)')
print('  FalsePositive  - detection was wrong -> tune the rule')
print('  Undetermined   - not enough evidence')

## Automation rules vs playbooks (exam)

| | Automation rules | Playbooks |
|-|-----------------|----------|
| **What** | Lightweight if/then logic | Full Logic App workflows |
| **Trigger** | Incident create/update, alert create | Called by automation rules or manually |
| **Actions** | Change severity, assign, run playbook, close | Any Azure/external API call |
| **Code** | No-code (portal UI) | Low-code (Logic Apps designer) |
| **Use case** | Triage automation, suppress noise | Complex response workflows |

- **Automation rules** decide *when* to run a playbook.
- **Playbooks** define *what actions* to take.
- Automation rules can also suppress, re-assign, and close incidents without playbooks.
- Limit: **512 automation rules** per workspace.

---

## What you built

- ✅ Data ingestion from 4+ sources (plus your own custom connector)
- ✅ Query engine with filter, aggregation, time window (KQL-like)
- ✅ Analytics rules, bad -> best progression, false-positive tuning
- ✅ MITRE ATT&CK coverage mapping
- ✅ Alert -> Incident correlation
- ✅ Full investigation workflow with automated playbooks
- ✅ Threat-intelligence watchlists / IOC matching

This is exactly what Microsoft Sentinel does - at massive scale with KQL, Logic Apps, and ML.

**Next lab**: [02 — Incident Response](../../02-incident-response/)